In [0]:
%sql
select * from workspace.silver.crm_customers

### Business Transformation and Modeling

In [0]:
%sql

SELECT 
       SHA2(CAST(ci.customer_id AS STRING), 256) AS customer_key,
       ci.customer_id,
       ci.first_name,
       ci.last_name,
       la.country,
       ci.marital_status,
       case 
           when ci.gender <> 'n/a' THEN ci.gender
           ELSE coalesce(ca.gender,'n/a')
       END as gender,
       ca.birth_date AS birthdate,
       ci.created_date AS create_date
FROM silver.crm_customers ci 
LEFT join silver.erp_customers ca
     ON ci.customer_number = ca.customer_number
LEFT join silver.erp_loc_a101 la
     ON ci.customer_number = la.customer_number




In [0]:
query = """
SELECT 
       SHA2(CAST(ci.customer_id AS STRING),256) AS customer_key,
       ci.customer_id,
       ci.first_name,
       ci.last_name,
       la.country,
       ci.marital_status,
       case 
           when ci.gender <> 'n/a' THEN ci.gender
           ELSE coalesce(ca.gender,'n/a')
       END as gender,
       ca.birth_date AS birthdate,
       ci.created_date AS create_date
FROM silver.crm_customers ci 
LEFT join silver.erp_customers ca
     ON ci.customer_number = ca.customer_number
LEFT join silver.erp_loc_a101 la
     ON ci.customer_number = la.customer_number


"""
df = spark.sql(query)

## Sanity checks

In [0]:
import pyspark.sql.functions as F
print("Sample Data:")
df.limit(10).display()
print("\nCheck for duplicate customer keys (should be 0):")
dup_Count=df.groupBy("customer_key").count().filter(F.col("count") > 1).count()

print("\n Gender Distribution:")
df.groupBy("gender").count().display()
print("\n Country Distribution:")
df.groupBy("country").count().orderBy("count",ascending=False).display()

### Write it to Gold Table

In [0]:
(
    df.write.mode("overwrite").format("delta").saveAsTable("gold.dim_customers")
)
print(f"Written {df.count()} rows to workspace.gold.dim_customers")

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customers